In [1]:
import pandas as pd
from tqdm.notebook import tqdm
from pprint import pprint

In [2]:
file_export = '../data/base_wiki_pt_cleaned_2.pq'

In [3]:
df = pd.read_parquet(file_export, engine='fastparquet')

In [4]:
df.texto = df.texto.str.strip()
df.texto = df.texto.str.replace(r'\s+', ' ', regex=True)

In [5]:
df.head()

,index,texto
0,1031610,O Opirus é um automóvel sedan de porte grande ...
1,1895765,Costa Macedo Giraldes Barba de Noronha e Brito...
2,649783,Eleutherodactylus cajamarcensis é uma espécie ...
3,1826843,Tricromatismo ou visão tricromática é a capaci...
4,1351999,Ode do grego antigo ōidē é um poema de estilo ...


In [7]:
def formar_paragrafos(text, n=200, p=0.75):
    paragrafos = text.split('.')
    novo_paragrafo = ''
    for x in range(len(paragrafos)):
        proposto = (novo_paragrafo + f' {paragrafos[x]}').strip()
        if(x == 0 and len(proposto) < n):
            novo_paragrafo = proposto
            continue
        if len(proposto) >= int(n*p) and len(proposto) <= n:
            novo_paragrafo = proposto
            break
    return novo_paragrafo, len(novo_paragrafo)

In [8]:
tqdm.pandas(desc='Podando parágrafos')
n, p = 200, 0.75
df[['text_cut', 'len_text_cut']] = df.apply(lambda x: formar_paragrafos(x['texto'], n, p), result_type='expand', axis=1)

In [154]:
df.head()

,index,texto,text_cut,len_text_cut
0,1031610,O Opirus é um automóvel sedan de porte grande ...,O Opirus é um automóvel sedan de porte grande ...,173
1,1895765,Costa Macedo Giraldes Barba de Noronha e Brito...,Costa Macedo Giraldes Barba de Noronha e Brito...,170
2,649783,Eleutherodactylus cajamarcensis é uma espécie ...,Eleutherodactylus cajamarcensis é uma espécie ...,82
3,1826843,Tricromatismo ou visão tricromática é a capaci...,Tricromatismo ou visão tricromática é a capaci...,151
4,1351999,Ode do grego antigo ōidē é um poema de estilo ...,Ode do grego antigo ōidē é um poema de estilo ...,169


In [155]:
print('% de parágrafos extraídos:', (len(df[df.len_text_cut > (n*p)]) / len(df)))

% de parágrafos extraídos: 0.824197


In [156]:
n*p

150.0

In [157]:
df_filtered = df[df.len_text_cut > (n*p)]

In [158]:
df_filtered = df_filtered.drop(columns=['texto'])

In [159]:
df_filtered.to_parquet('data/base_wiki_pt_cleaned_cutted.pq', index=False)
df_filtered.head(10)

,index,text_cut,len_text_cut
0,1031610,O Opirus é um automóvel sedan de porte grande ...,173
1,1895765,Costa Macedo Giraldes Barba de Noronha e Brito...,170
3,1826843,Tricromatismo ou visão tricromática é a capaci...,151
4,1351999,Ode do grego antigo ōidē é um poema de estilo ...,169
7,1396393,Palinopsia grego: palin para novamente e opsia...,186
8,1911589,Sir Walter Lawry Buller 9 de Outubro de 1838 -...,176
10,1151976,Nome usado na lista do Património Mundial A pa...,180
11,1422950,Paulo em latim: Paulus; em grego:; romaniz O ...,156
12,1599893,Rua do Riachuelo hoje Rua Riachuelo é um logra...,185
13,663813,"A torção de talheres, ou ato de entortar colhe...",196


In [11]:
df_filtered = pd.read_parquet('../data/base_wiki_pt_cleaned_cutted.pq')

In [12]:
indexes = df_filtered.sample(frac=1, random_state=42).index.tolist()
perc_train, perc_eval, perc_test = 0.8, .015, 0.215
qtde_train, qtde_eval, qtde_test = int(len(indexes)*perc_train), int(len(indexes)*perc_eval), int(len(indexes)*perc_test)
indexes_train = indexes[:qtde_train]
indexes_eval = indexes[qtde_train: qtde_train+qtde_eval]
indexes_test = indexes[qtde_train+qtde_eval: -1]

In [7]:
1-(0.8-.015)

0.21499999999999997

In [8]:
len(indexes_train), len(indexes_eval), len(indexes_test)

(659357, 12362, 152477)

In [9]:
df_train = df_filtered[df_filtered.index.isin(indexes_train)]
df_eval = df_filtered[df_filtered.index.isin(indexes_eval)]
df_test = df_filtered[df_filtered.index.isin(indexes_test)]

df_train.to_parquet('data/train_wiki_cleaned_cutted.pq', index=False)
df_eval.to_parquet('data/eval_wiki_cleaned_cutted.pq', index=False)
df_test.to_parquet('data/test_wiki_cleaned_cutted.pq', index=False)

In [2]:
df_train = pd.read_parquet('/media/alvarinho/dados/Estudos/data/train_wiki_cleaned_cutted.pq')
df_eval = pd.read_parquet('/media/alvarinho/dados/Estudos/data/eval_wiki_cleaned_cutted.pq')
df_test = pd.read_parquet('/media/alvarinho/dados/Estudos/data/test_wiki_cleaned_cutted.pq')

In [3]:
text_data = df_train['text_cut'].values.tolist()

In [4]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import TemplateProcessing

tokenizers = {}
for vocab_size in [500, 1000, 5000, 10000, 25000, 50000]:
    bpe_model = BPE()
    tok = Tokenizer(bpe_model)
    tok.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=3,
        special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"],
        limit_alphabet=1000,
        initial_alphabet=list("0123456789"),
        continuing_subword_prefix="##",
        show_progress=True)

    tok.train_from_iterator(text_data, trainer=trainer)
    tok.post_processor = TemplateProcessing(
        single="[BOS] $A [EOS]",
        special_tokens=[
            ("[BOS]", tok.token_to_id("[BOS]")),
            ("[EOS]", tok.token_to_id("[EOS]")),
        ],
    )
    
    tokenizers[vocab_size] = tok

In [5]:
def tokenize(x, tokenizer: Tokenizer):
    return len(tokenizer.encode(x).ids)

In [6]:
for vocab_size, tokenizer in tokenizers.items():
    df_train[f'len_tokenizer_{vocab_size}'] = df_train.apply(lambda x: 1 - (tokenize(x['text_cut'], tokenizer) / x['len_text_cut']), axis=1)

In [8]:
df_train.describe()

,index,len_text_cut,len_tokenizer_500,len_tokenizer_1000,len_tokenizer_5000,len_tokenizer_10000,len_tokenizer_25000,len_tokenizer_50000
count,6.593570e+05,659357.000000,659357.000000,659357.000000,659357.000000,659357.000000,659357.000000,659357.000000
mean,1.017740e+06,176.482673,0.152411,0.450985,0.698897,0.735765,0.766738,0.781251
std,5.492166e+05,14.386798,0.016572,0.031738,0.039950,0.035283,0.029714,0.025950
min,2.000000e+00,151.000000,0.010638,0.021277,0.153846,0.205128,0.276923,0.302564
25%,5.469430e+05,164.000000,0.141414,0.431138,0.674286,0.715084,0.750000,0.766839
50%,1.010603e+06,177.000000,0.152866,0.451613,0.700637,0.739130,0.770053,0.783784
75%,1.489335e+06,189.000000,0.163522,0.471338,0.726257,0.760736,0.786982,0.798780
max,2.060585e+06,200.000000,0.324176,0.566879,0.832432,0.844828,0.866242,0.872611
